# HW2 Part 1: data cleaning and validation

Run from `Homeworks/HW2/Quentin_Benchetrit/`. The notebook works on a local copy of the database, `data/data.db` (gitignored), copied from `Lectures/Lecture 4/data/data.db` on the first run. It writes `ohlc_hw_cleaned` and never modifies `ohlc_hw`.

The supplied `ohlc` table is used as a reference for the trading calendar, for the adjustment factor `adj_close / close` of each date, and as evidence when a value looks wrong.

In [1]:
import html
import shutil
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 200)

DB_PATH = Path('data/data.db')
RAW_PRICES = ['open', 'high', 'low', 'close']
PRICES = RAW_PRICES + ['adj_close']
VALUES = PRICES + ['volume']
KEY = ['ticker', 'ts']
PRICE_TOL = 1e-6  # relative difference above which a price disagrees with the reference

if not DB_PATH.is_file():
    course_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file()), None)
    source = course_root / 'Lectures/Lecture 4/data/data.db' if course_root else None
    if source is None or not source.is_file():
        raise FileNotFoundError('Download data.db from bCourses into Lectures/Lecture 4/data/')
    DB_PATH.parent.mkdir(exist_ok=True)
    shutil.copy2(source, DB_PATH)

conn = sqlite3.connect(DB_PATH)

## 1. Load and inspect the data

In [2]:
raw = pd.read_sql_query('SELECT * FROM ohlc_hw', conn)
reference = pd.read_sql_query('SELECT * FROM ohlc', conn)
print(f'ohlc_hw: {len(raw):,} rows | ohlc: {len(reference):,} rows\n')
raw.info()

ohlc_hw: 397,711 rows | ohlc: 368,251 rows

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 397711 entries, 0 to 397710
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   ts         397711 non-null  object 
 1   open       397711 non-null  float64
 2   high       397711 non-null  float64
 3   low        397711 non-null  float64
 4   close      397711 non-null  float64
 5   adj_close  377856 non-null  float64
 6   volume     397711 non-null  int64  
 7   ticker     397711 non-null  object 
dtypes: float64(5), int64(1), object(2)
memory usage: 24.3+ MB


In [3]:
display(raw.describe())
display(raw.head())

,open,high,low,close,adj_close,volume
count,397711.000000,397711.000000,397711.000000,397711.000000,377856.000000,3.977110e+05
mean,46.285479,46.747660,45.816279,46.292903,51.652250,3.097773e+07
std,67.132594,67.778210,66.477178,67.141879,194.584045,1.266237e+08
min,0.000000,0.005208,0.004801,0.005208,0.002945,0.000000e+00
25%,4.209938,4.240000,4.160000,4.200000,1.390000,1.730700e+06
50%,19.850000,20.060000,19.630000,19.860000,11.960000,4.648200e+06
75%,57.530000,58.130000,56.950000,57.530000,43.770000,1.102380e+07
max,604.260000,608.630000,597.880000,605.400000,10442.510783,9.276606e+09


,ts,open,high,low,close,adj_close,volume,ticker
0,1975-04-14,5.4300,5.4800,5.3600,5.4400,1.640000,446392,CAT
1,1978-04-05,4.8400,4.8900,4.8100,4.8800,0.656990,355200,CVX
2,2018-05-25,124.5400,124.7600,121.2200,122.1900,92.220000,8936649,CVX
3,1995-12-07,9.9700,9.9700,9.8300,9.9400,5.880000,3954150,HD
4,1981-07-23,0.1038,0.1038,0.1038,0.1038,0.080013,34451232,AAPL


| Column | Meaning |
|---|---|
| `ts` | Date of the trading session |
| `ticker` | Vendor label of the stock |
| `open`, `high`, `low`, `close` | Raw prices of the session |
| `adj_close` | Close adjusted for splits and dividends, used for returns |
| `volume` | Shares traded |

In [4]:
def parse_dates(values):
    """Date strings to timestamps; unparseable values become NaT."""
    return pd.to_datetime(values, errors='coerce', format='ISO8601').dt.normalize()


def normalize_ticker(labels):
    """Vendor label to ticker: decode HTML entities, drop tags, trim whitespace, upper-case."""
    text = labels.astype('string').map(html.unescape, na_action='ignore').astype('string')
    return text.str.replace(r'<[^>]+>', '', regex=True).str.strip().str.upper()


raw_dates = parse_dates(raw['ts'])
raw_tickers = normalize_ticker(raw['ticker'])
ref_tickers = set(reference['ticker'])
print(f'Dates: {raw_dates.min():%Y-%m-%d} to {raw_dates.max():%Y-%m-%d}, unparseable: {raw_dates.isna().sum()}')
print(f'{raw["ticker"].nunique()} raw labels for {raw_tickers.nunique()} tickers, '
      f'not in the reference: {sorted(set(raw_tickers.dropna()) - ref_tickers) or "none"}')

label_audit = (pd.DataFrame({'raw_label': raw['ticker'], 'ticker': raw_tickers}).value_counts()
               .rename('rows').reset_index().sort_values(['ticker', 'rows'], ascending=[True, False], ignore_index=True))
with pd.option_context('display.max_colwidth', 120):
    display(label_audit.loc[label_audit['ticker'] == label_audit['ticker'].iloc[0]])
display(pd.DataFrame({'ticker': raw_tickers, 'ts': raw_dates}).groupby('ticker')['ts']
        .agg(first_date='min', last_date='max', rows='size'))

Dates: 1962-01-02 to 2024-12-06, unparseable: 0
150 raw labels for 30 tickers, not in the reference: none


,raw_label,ticker,rows
0,AAPL,AAPL,10747
1,"<a target=""_blank""><span style=""color: red"">AAPL</span></a>",AAPL,318
2,"<a target=""_blank""><span style=""color: blue"">AAPL</span></a>",AAPL,307
3,"<a target=""_blank""><span style=""font-weight: bold; color: blue"">AAPL</span></a>",AAPL,306
4,"<a target=""_blank""><span style=""font-weight: bold; color: red"">AAPL</span></a>",AAPL,264


,first_date,last_date,rows
ticker,,,
AAPL,1980-12-12,2024-10-18,11942
AMGN,1983-06-17,2024-10-18,11290
AMZN,1997-05-15,2024-12-06,7458
AXP,1972-04-03,2024-10-18,14260
BA,1962-01-02,2024-10-18,16980
CAT,1962-01-02,2024-10-18,17044
CRM,2004-06-23,2024-10-18,5525
CSCO,1990-02-16,2024-10-18,9427
CVX,1962-01-02,2024-12-06,17073


## 2. Detect data quality issues

Everything in this section runs before any repair. Tickers are normalized and exact duplicates set aside first, since grouped returns and calendar checks make no sense on split or repeated series.

### 2.1 Duplicates

In [5]:
def prepare_keys(frame):
    """Normalized ticker and parsed date; the delivered strings are kept as ticker_raw and ts_raw."""
    out = frame.rename(columns={'ticker': 'ticker_raw', 'ts': 'ts_raw'})
    out['ticker'] = normalize_ticker(out['ticker_raw'])
    out['ts'] = parse_dates(out['ts_raw'])
    return out


def split_duplicates(frame):
    """Exact copies (same key and values) and keys with conflicting versions."""
    exact_extra = frame.duplicated(KEY + VALUES)
    versions = frame.loc[~exact_extra].groupby(KEY).size()
    return exact_extra, versions[versions > 1].index


keyed = prepare_keys(raw)
exact_extra, conflicting_keys = split_duplicates(keyed)
print(f'Exact duplicates: {exact_extra.sum():,} | conflicting keys: {len(conflicting_keys)}')

deduped = keyed.loc[~exact_extra]
known = deduped['ts'].notna() & deduped['ticker'].isin(ref_tickers)
in_conflict = pd.MultiIndex.from_frame(deduped[KEY]).isin(conflicting_keys)
diag = deduped.loc[known & ~in_conflict].sort_values(KEY).reset_index(drop=True)

Exact duplicates: 29,460 | conflicting keys: 0


### 2.2 Missing values

In [6]:
def missing_by_column(frame):
    """Number of missing cells in each column."""
    return frame[VALUES].isna().sum()


display(pd.DataFrame({'raw delivery': missing_by_column(raw), 'unique records': missing_by_column(diag)}))

,raw delivery,unique records
open,0,0
high,0,0
low,0,0
close,0,0
adj_close,19855,18374
volume,0,0


### 2.3 Missing trading days

The expected sessions of a ticker are its dates in `ohlc`, over the history the reference gives for it. Business weekdays would count holidays as missing. The detector is first checked on five sessions with one removed.

In [7]:
def missing_sessions(frame, expected_dates):
    """Expected dates absent from one ticker's rows."""
    observed = pd.DatetimeIndex(frame['ts'].dropna().unique())
    return pd.DatetimeIndex(expected_dates).difference(observed).sort_values()


ref_calendar = pd.DataFrame({'ticker': reference['ticker'], 'ts': parse_dates(reference['ts'])})

demo = ref_calendar.loc[ref_calendar['ticker'] == 'AAPL', 'ts'].sort_values().iloc[:5]
demo_gap = missing_sessions(demo.drop(demo.index[2]).to_frame(), demo)
print(f'Removed {demo.iloc[2]:%Y-%m-%d}, detected {[f"{d:%Y-%m-%d}" for d in demo_gap]}\n')
assert list(demo_gap) == [demo.iloc[2]]

observed = dict(tuple(deduped.loc[known].groupby('ticker')['ts']))
missing_dates = []
for ticker, expected in ref_calendar.groupby('ticker')['ts']:
    gaps = missing_sessions(observed.get(ticker, pd.Series(dtype='datetime64[ns]')).to_frame('ts'), expected)
    missing_dates += [(ticker, day) for day in gaps]
    print(f'Ticker {ticker} is missing {len(gaps)} trading days between {expected.min():%Y-%m-%d} and {expected.max():%Y-%m-%d}')
missing_dates = pd.DataFrame(missing_dates, columns=KEY)

Removed 1980-12-16, detected ['1980-12-16']

Ticker AAPL is missing 0 trading days between 1980-12-12 and 2024-10-18
Ticker AMGN is missing 0 trading days between 1983-06-17 and 2024-10-18
Ticker AMZN is missing 0 trading days between 1997-05-15 and 2024-12-06
Ticker AXP is missing 0 trading days between 1972-04-03 and 2024-10-18
Ticker BA is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker CAT is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker CRM is missing 0 trading days between 2004-06-23 and 2024-10-18
Ticker CSCO is missing 0 trading days between 1990-02-16 and 2024-10-18
Ticker CVX is missing 0 trading days between 1962-01-02 and 2024-12-06
Ticker DIS is missing 0 trading days between 1962-01-02 and 2024-12-06
Ticker GS is missing 0 trading days between 1999-05-04 and 2024-10-18
Ticker HD is missing 0 trading days between 1981-09-22 and 2024-10-18
Ticker HON is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker IBM is missing 0 trading

### 2.4 Extreme returns

Simple returns `(close - prev_close) / prev_close` within each ticker, flagged when `abs(return - mean_return) > 3 * std_return` with the ticker's mean and sample standard deviation. Undefined returns (first row, missing or nonpositive close) are left out, and a ticker with an undefined or zero standard deviation gets no flag. This is a review flag, not a deletion rule.

In [8]:
def flag_extreme_returns(frame, price_col='close', k=3.0):
    """Simple returns and k-sigma flags within each ticker (sample std, both tails)."""
    data = frame.sort_values(KEY)
    price = data[price_col].where(np.isfinite(data[price_col]) & data[price_col].gt(0))
    prev = price.groupby(data['ticker']).shift(1)
    ret = (price - prev) / prev
    grouped = ret.groupby(data['ticker'])
    mean, std = grouped.transform('mean'), grouped.transform('std')
    usable = std.gt(0)  # False for an undefined or zero standard deviation
    flag = ret.notna() & usable & (ret - mean).abs().gt(k * std)
    return pd.DataFrame({'return': ret, 'mean_return': mean, 'std_return': std,
                         'z_score': (ret - mean) / std.where(usable), 'extreme': flag}, index=data.index)


screen = flag_extreme_returns(diag)
no_dispersion = diag.loc[~screen['std_return'].gt(0), 'ticker'].unique()
print(f'Flagged: {int(screen["extreme"].sum()):,} of {int(screen["return"].count()):,} returns')
print(f'Tickers with a zero or undefined standard deviation: {list(no_dispersion) or "none"}')

Flagged: 5,056 of 368,221 returns
Tickers with a zero or undefined standard deviation: none


### 2.5 Prices, OHLC bounds and volume

Raw prices must be positive, finite and satisfy `low <= open <= high` and `low <= close <= high`. Adjusted close is only checked for being positive and finite.

In [9]:
def price_checks(frame):
    """One boolean column per check."""
    o, h, l, c = (frame[col] for col in RAW_PRICES)
    v = frame['volume']
    return pd.DataFrame({
        'missing_raw_price': frame[RAW_PRICES].isna().any(axis=1),
        'missing_adj_close': frame['adj_close'].isna(),
        'nonfinite_price': np.isinf(frame[PRICES]).any(axis=1),
        'nonpositive_price': frame[PRICES].le(0).any(axis=1),
        'high_below_close': h.lt(c),
        'low_above_close': l.gt(c),
        'open_outside_range': o.lt(l) | o.gt(h),
        'high_below_low': h.lt(l),
        'missing_volume': v.isna(),
        'invalid_volume': np.isinf(v) | v.lt(0),
        'zero_volume': v.eq(0),
    })


def true_columns(mask):
    """Names of the True columns of each row, comma-separated."""
    names = np.array(mask.columns)
    return pd.Series([', '.join(names[row]) for row in mask.to_numpy()], index=mask.index, dtype=object)


checks = price_checks(diag)
display(checks.sum().rename('rows').to_frame().T)

,missing_raw_price,missing_adj_close,nonfinite_price,nonpositive_price,high_below_close,low_above_close,open_outside_range,high_below_low,missing_volume,invalid_volume,zero_volume
rows,0,18374,0,5,0,0,5,0,0,0,75


### 2.6 Comparison with the reference

In [10]:
def compare_with_reference(frame, reference):
    """Reference prices of each row's (ticker, ts), whether the raw prices differ, and the factor check."""
    ref = reference.assign(ts=parse_dates(reference['ts'])).set_index(KEY)[PRICES].add_prefix('ref_')
    keys = pd.MultiIndex.from_arrays([frame['ticker'].astype(object), frame['ts']])
    out = ref.reindex(keys).set_axis(frame.index)
    with np.errstate(invalid='ignore', divide='ignore'):  # a zero price on both sides gives 0/0
        relative = frame[RAW_PRICES].to_numpy() / out[[f'ref_{c}' for c in RAW_PRICES]].to_numpy() - 1
    out['raw_differs'] = (np.abs(relative) > PRICE_TOL).any(axis=1)
    out['ref_factor'] = out['ref_adj_close'] / out['ref_close']
    out['factor_differs'] = (frame['adj_close'] / frame['close'] / out['ref_factor'] - 1).abs().gt(PRICE_TOL)
    return out


evidence = compare_with_reference(diag, reference)
ratio = (diag['adj_close'] / (diag['close'] * evidence['ref_factor']))[evidence['factor_differs']]
print(f'Rows whose raw prices differ from the reference: {evidence["raw_differs"].sum()}')
print(f'adj_close inconsistent with the reference factor: {len(ratio):,} rows, '
      f'{ratio.min():.1f} to {ratio.max():.1f} times the expected value')

Rows whose raw prices differ from the reference: 0
adj_close inconsistent with the reference factor: 10,417 rows, 4.0 to 24.0 times the expected value


The raw prices are identical to the reference, so any problem in them is also in the reference. The delivered adjusted close is the exception: on 10,417 rows it is 4 to 24 times too large.

### 2.7 Data quality report

One row per issue and observation; an observation can have several issues, and every missing session is its own row. Label formatting is not tied to a date, so it gets one row per raw label.

| Severity | Rule | Issue types |
|---|---|---|
| high | The value is wrong and would distort returns | conflicting duplicates, nonpositive or non-finite prices, impossible OHLC relationships, adjusted close inconsistent with the reference, unparseable dates, unknown tickers |
| medium | Needs a repair or a review, but the fix is clear or the value may be genuine | missing prices, extreme returns, missing sessions, invalid volume |
| low | No effect on prices once handled | label formatting, exact duplicates, zero or missing volume |

In [11]:
SEVERITY = {
    'conflicting_duplicate': 'high', 'nonpositive_price': 'high', 'nonfinite_price': 'high',
    'high_below_close': 'high', 'low_above_close': 'high', 'open_outside_range': 'high', 'high_below_low': 'high',
    'adj_close_inconsistent': 'high', 'unparseable_date': 'high', 'unknown_ticker': 'high',
    'missing_raw_price': 'medium', 'missing_adj_close': 'medium', 'extreme_return': 'medium',
    'missing_session': 'medium', 'invalid_volume': 'medium',
    'ticker_label_format': 'low', 'exact_duplicate': 'low', 'zero_volume': 'low', 'missing_volume': 'low',
}


def issue_rows(frame, mask, issue_type, describe):
    """Report rows for one check; `describe` is a string or a function of the selected rows."""
    selected = frame.loc[mask]
    return pd.DataFrame({'ticker': selected['ticker'], 'issue_type': issue_type, 'issue_date': selected['ts'],
                         'description': describe(selected) if callable(describe) else describe,
                         'severity': SEVERITY[issue_type]})


def fmt(values, spec='{:.4f}'):
    return values.map(spec.format).astype(object)


rows = diag.join(screen).join(evidence)
per_key = keyed.groupby(KEY).size().rename('records').reset_index()
labels = label_audit.loc[label_audit['raw_label'] != label_audit['ticker']]
parts = [
    pd.DataFrame({'ticker': labels['ticker'], 'issue_type': 'ticker_label_format', 'issue_date': pd.NaT,
                  'description': labels['rows'].astype(str) + ' rows labelled ' + labels['raw_label'].map(repr),
                  'severity': 'low'}),
    issue_rows(keyed, keyed['ts'].isna(), 'unparseable_date', 'date cannot be parsed'),
    issue_rows(keyed, ~keyed['ticker'].isin(ref_tickers), 'unknown_ticker', 'ticker not in the reference'),
    issue_rows(keyed.loc[exact_extra], slice(None), 'exact_duplicate', 'exact copy of another record'),
    issue_rows(per_key, pd.MultiIndex.from_frame(per_key[KEY]).isin(conflicting_keys), 'conflicting_duplicate',
               'several versions of this record'),
    issue_rows(rows, checks['missing_raw_price'], 'missing_raw_price', 'raw price missing'),
    issue_rows(rows, checks['missing_adj_close'], 'missing_adj_close', 'adj_close missing'),
    issue_rows(rows, checks['nonfinite_price'], 'nonfinite_price', 'non-finite price'),
    issue_rows(rows, checks['nonpositive_price'], 'nonpositive_price', lambda s: true_columns(s[PRICES].le(0)) + ' <= 0'),
    issue_rows(rows, checks['high_below_close'], 'high_below_close',
               lambda s: 'high ' + fmt(s['high']) + ' < close ' + fmt(s['close'])),
    issue_rows(rows, checks['low_above_close'], 'low_above_close',
               lambda s: 'low ' + fmt(s['low']) + ' > close ' + fmt(s['close'])),
    issue_rows(rows, checks['open_outside_range'], 'open_outside_range',
               lambda s: 'open ' + fmt(s['open']) + ' outside [' + fmt(s['low']) + ', ' + fmt(s['high']) + ']'),
    issue_rows(rows, checks['high_below_low'], 'high_below_low', 'high below low'),
    issue_rows(rows, checks['missing_volume'], 'missing_volume', 'volume missing'),
    issue_rows(rows, checks['invalid_volume'], 'invalid_volume', 'negative or non-finite volume'),
    issue_rows(rows, checks['zero_volume'], 'zero_volume', 'zero volume'),
    issue_rows(rows, rows['extreme'], 'extreme_return',
               lambda s: 'return ' + fmt(s['return'], '{:+.2%}') + ', z = ' + fmt(s['z_score'], '{:+.1f}')),
    issue_rows(rows, rows['factor_differs'], 'adj_close_inconsistent',
               lambda s: 'adj_close ' + fmt(s['adj_close']) + ', expected ' + fmt(s['close'] * s['ref_factor'])),
    issue_rows(missing_dates, slice(None), 'missing_session', 'session absent from the delivery'),
]
quality_report = pd.concat([p for p in parts if len(p)], ignore_index=True)
print(f'{len(quality_report):,} issues')
display(quality_report.groupby(['severity', 'issue_type']).size().rename('issues').to_frame())
display(quality_report.groupby('issue_type').head(1))

63,512 issues


issues
severity issue_type                    
high     adj_close_inconsistent   10417
         nonpositive_price            5
         open_outside_range           5
low      exact_duplicate          29460
         ticker_label_format        120
         zero_volume                 75
medium   extreme_return            5056
         missing_adj_close        18374

,ticker,issue_type,issue_date,description,severity
0,AAPL,ticker_label_format,NaT,"318 rows labelled '<a target=""_blank""><span st...",low
120,DIS,exact_duplicate,2004-04-28,exact copy of another record,low
29580,AAPL,missing_adj_close,1981-01-08,adj_close missing,medium
47954,CVX,nonpositive_price,1962-01-02,open <= 0,high
47959,CVX,open_outside_range,1962-01-02,"open 0.0000 outside [3.2400, 3.3000]",high
47964,AXP,zero_volume,1972-12-14,zero volume,low
48039,AAPL,extreme_return,1980-12-26,"return +9.23%, z = +3.3",medium
53095,AAPL,adj_close_inconsistent,1981-02-03,"adj_close 1.2646, expected 0.0951",high


### 2.8 Detectors on injected examples

The cases the delivery does not contain are checked on small frames where the answer is known.

In [12]:
injected = pd.DataFrame({
    'ticker': 'DEMO', 'ts': pd.bdate_range('2024-01-02', periods=6),
    'open':      [10.0, 10.2, 10.1, 10.3, 10.0, 10.2],
    'high':      [10.5, 10.4,  9.9, 10.6, 10.4, 10.5],
    'low':       [ 9.8, 10.0,  9.7, 10.1, 10.6, 10.0],
    'close':     [10.1, 10.3, 10.0, np.nan, 10.2, -1.0],
    'adj_close': [10.0, 10.2, 9.9, np.inf, 10.1, 10.1],
    'volume':    [100.0, 0.0, -5.0, np.nan, 100.0, 100.0],
})
found = price_checks(injected)
expected = {'missing_raw_price': [3], 'nonfinite_price': [3], 'missing_volume': [3], 'zero_volume': [1],
            'high_below_close': [2], 'open_outside_range': [2, 4], 'invalid_volume': [2],
            'low_above_close': [4, 5], 'high_below_low': [4], 'nonpositive_price': [5]}
for check, positions in expected.items():
    assert found.index[found[check]].tolist() == positions, check
display(injected.assign(detected=true_columns(found)))

# 3-sigma screen: a spike, a lasting drop (lower tail) and a constant series (zero std).
rng = np.random.default_rng(0)
calm = 100 * np.exp(np.cumsum(rng.normal(0, 0.01, 40)))
spike, drop = calm.copy(), calm.copy()
spike[20] *= 10
drop[25:] *= 0.7
series = pd.concat([pd.DataFrame({'ticker': name, 'ts': pd.bdate_range('2024-01-02', periods=len(p)), 'close': p})
                    for name, p in {'SPIKE': spike, 'DROP': drop, 'FLAT': np.full(10, 50.0)}.items()],
                   ignore_index=True)
flags = series.join(flag_extreme_returns(series))
display(flags.loc[flags['extreme'], ['ticker', 'ts', 'return', 'z_score']])
assert set(flags.loc[flags['extreme'], 'ticker']) == {'SPIKE', 'DROP'}
assert flags.loc[flags['extreme'] & flags['ticker'].eq('DROP'), 'return'].lt(0).all()
assert not flags.loc[flags['ticker'].eq('FLAT'), 'extreme'].any()

,ticker,ts,open,high,low,close,adj_close,volume,detected
0,DEMO,2024-01-02,10.0,10.5,9.8,10.1,10.0,100.0,
1,DEMO,2024-01-03,10.2,10.4,10.0,10.3,10.2,0.0,zero_volume
2,DEMO,2024-01-04,10.1,9.9,9.7,10.0,9.9,-5.0,"high_below_close, open_outside_range, invalid_..."
3,DEMO,2024-01-05,10.3,10.6,10.1,NaN,inf,NaN,"missing_raw_price, nonfinite_price, missing_vo..."
4,DEMO,2024-01-08,10.0,10.4,10.6,10.2,10.1,100.0,"low_above_close, open_outside_range, high_belo..."
5,DEMO,2024-01-09,10.2,10.5,10.0,-1.0,10.1,100.0,"nonpositive_price, low_above_close"


,ticker,ts,return,z_score
20,SPIKE,2024-01-30,8.987155,6.054842
65,DROP,2024-02-06,-0.299342,-6.000375


## 3. Clean the data

Rows are sorted by ticker and date, every changed value is flagged, and a row that cannot be repaired defensibly is quarantined with a reason.

| Step | Rule |
|---|---|
| 1 | Unknown tickers and unparseable dates are quarantined |
| 2 | Exact duplicates: one copy kept |
| 3 | Conflicting keys: keep the version whose raw prices match the reference, otherwise quarantine all versions |
| 4 | Rows with all four raw prices missing are dropped, before any filling |
| 5 | Nonpositive or non-finite raw prices are treated as missing. Missing raw prices are forward-filled within the ticker; a carried open or close outside today's range becomes the midpoint of high and low, and a carried high or low is widened to contain the open and close. Leading gaps are quarantined |
| 6 | Confirmed errors (one-day spikes, see 3.2) are interpolated between the neighbouring sessions, with no extrapolation at the edges |
| 7 | Remaining impossible OHLC rows: Option 3, keep open and close and widen high and low to contain them |
| 8 | Missing or inconsistent `adj_close`: close times the reference factor of the same date, never carried to another date |
| 9 | Missing or invalid volume: 0 with a flag (unknown, not "no trading"); zero volume is flagged |

### 3.1 Functions

In [13]:
def enforce_range(values, changed):
    """For estimated cells: an open/close outside the range moves to the midpoint of high and low,
    and an estimated high/low is widened to contain the open and close."""
    values = values.copy()
    midpoint = (values['high'] + values['low']) / 2
    for col in ['open', 'close']:
        outside = changed[col] & ~values[col].between(values['low'], values['high'])
        values.loc[outside, col] = midpoint[outside]
    values['high'] = values['high'].where(~changed['high'], values[['open', 'close', 'high']].max(axis=1, skipna=False))
    values['low'] = values['low'].where(~changed['low'], values[['open', 'close', 'low']].min(axis=1, skipna=False))
    return values


def invalidate_bad_prices(frame):
    """Nonpositive or non-finite raw prices become missing."""
    bad = frame[RAW_PRICES].notna() & (np.isinf(frame[RAW_PRICES]) | frame[RAW_PRICES].le(0))
    out = frame.copy()
    out[RAW_PRICES] = out[RAW_PRICES].mask(bad)
    return out, bad


def fill_missing_prices(frame):
    """Forward fill within each ticker (frame sorted by ticker and date), then the range check.
    A leading gap has nothing to carry and stays missing."""
    data = frame.copy()
    values = data[RAW_PRICES].fillna(data.groupby('ticker')[RAW_PRICES].ffill())
    changed = data[RAW_PRICES].isna() & values.notna()
    data[RAW_PRICES] = enforce_range(values, changed)
    return data, changed


def interpolate_errors(frame, bad):
    """Linear interpolation between valid neighbours of the same ticker for the cells in `bad`.
    Errors at the start or end of a history stay missing."""
    data = frame.copy()
    values = data[RAW_PRICES].mask(bad)
    inside = values.groupby(data['ticker']).transform(lambda s: s.interpolate(limit_area='inside'))
    values = values.where(~bad, inside)
    data[RAW_PRICES] = enforce_range(values, bad & values.notna())
    return data


def widen_ranges(frame):
    """Option 3: keep open and close, widen high and low just enough to contain them."""
    data = frame.copy()
    high = data[['open', 'high', 'close']].max(axis=1)
    low = data[['open', 'low', 'close']].min(axis=1)
    changed = high.ne(data['high']) | low.ne(data['low'])
    data['high'], data['low'] = high, low
    return data, changed


def resolve_conflicts(frame, evidence):
    """Keep the only version of a repeated key whose raw prices match the reference; quarantine otherwise."""
    repeated = frame.duplicated(KEY, keep=False)
    kept, quarantined = [], []
    for _, versions in frame.loc[repeated].groupby(KEY):
        matches = ~evidence.loc[versions.index, 'raw_differs']
        if matches.sum() == 1:
            kept.append(matches.idxmax())
        else:
            quarantined += list(versions.index)
    out = frame.loc[~repeated | frame.index.isin(kept)].copy()
    out['conflict_resolved'] = out.index.isin(kept)
    return out, frame.loc[quarantined]


def rebuild_adj_close(frame, factor, need):
    """adj_close = close x factor of the same date for the rows in `need`; NaN where there is no factor."""
    data = frame.copy()
    usable = need & np.isfinite(factor) & factor.gt(0)
    data.loc[usable, 'adj_close'] = data.loc[usable, 'close'] * factor[usable]
    data.loc[need & ~usable, 'adj_close'] = np.nan
    return data, usable


def neighbour_factor(frame, factor, rows, tol=0.005):
    """Mean factor of the previous and next sessions, only where those two agree (no corporate action between)."""
    grouped = factor.groupby(frame['ticker'])
    before, after = grouped.shift(1), grouped.shift(-1)
    return ((before + after) / 2).where(rows & (after / before - 1).abs().le(tol))


def repair_volume(frame):
    """Missing, negative or non-finite volume becomes 0 with a flag; observed zeros are only flagged."""
    data = frame.copy()
    unknown = data['volume'].isna() | np.isinf(data['volume']) | data['volume'].lt(0)
    data['zero_volume'] = data['volume'].eq(0)
    data['volume_repaired'] = unknown
    data.loc[unknown, 'volume'] = 0
    return data

In [14]:
# Steps 4 and 5: the empty row is dropped before filling; 100 is outside [109, 111] so the close becomes
# the midpoint; the carried high of C is widened to its close; B has a leading gap.
ff = pd.DataFrame({
    'ticker': ['A', 'A', 'A', 'B', 'B', 'C', 'C'],
    'ts': pd.to_datetime(['2024-01-02', '2024-01-03', '2024-01-04', '2024-01-02', '2024-01-03', '2024-01-02', '2024-01-03']),
    'open':  [99.5, np.nan, 109.5, np.nan, 50.0, 20.0, 22.0],
    'high':  [101.0, np.nan, 111.0, 51.0, 52.0, 21.0, np.nan],
    'low':   [99.0, np.nan, 109.0, 49.0, 49.5, 19.0, 21.5],
    'close': [100.0, np.nan, np.nan, np.nan, 51.0, 20.5, 22.5],
})
ff = ff.loc[~ff[RAW_PRICES].isna().all(axis=1)]
ff_filled, _ = fill_missing_prices(ff)
display(ff_filled)
assert 1 not in ff_filled.index and ff_filled.loc[2, 'close'] == 110.0
assert ff_filled.loc[3, ['open', 'close']].isna().all() and ff_filled.loc[6, 'high'] == 22.5

# Step 5: nonpositive and infinite closes are refilled from the last valid close.
inv = pd.DataFrame({'ticker': 'A', 'ts': pd.bdate_range('2024-01-02', periods=3), 'open': [10.0, 10.2, 10.1],
                    'high': [10.5, 10.6, 10.4], 'low': [9.8, 10.0, 9.9], 'close': [10.2, -10.3, np.inf]})
assert fill_missing_prices(invalidate_bad_prices(inv)[0])[0]['close'].tolist() == [10.2, 10.2, 10.2]

# Step 6: an interior error is interpolated, an error on the last day is not extrapolated.
ie = pd.DataFrame({'ticker': 'A', 'ts': pd.bdate_range('2024-02-01', periods=5), 'open': [100.0, 101, 102, 103, 104],
                   'high': [101.0, 102, 103, 104, 105], 'low': [99.0, 100, 101, 102, 103], 'close': [100.0, 101, 1020, 103, 1040]})
bad = pd.DataFrame(False, index=ie.index, columns=RAW_PRICES)
bad.loc[[2, 4], 'close'] = True
ie_fixed = interpolate_errors(ie, bad)
assert ie_fixed.loc[2, 'close'] == 102.0 and np.isnan(ie_fixed.loc[4, 'close'])

# Step 7: a close above the reported high widens the range.
assert widen_ranges(pd.DataFrame({'open': [10.0], 'high': [10.2], 'low': [9.9], 'close': [10.25]}))[0].loc[0, 'high'] == 10.25

# Step 3: B keeps the version matching the reference, C has no evidence and is quarantined.
ck = pd.DataFrame({'ticker': list('BBCC'), 'ts': pd.Timestamp('2024-03-01'), 'close': [20.5, 22.0, 30.5, 30.8]})
ck_kept, ck_out = resolve_conflicts(ck, pd.DataFrame({'raw_differs': [False, True, True, True]}))
assert ck_kept.index.tolist() == [0] and ck_out.index.tolist() == [2, 3]

# Step 8: each date uses its own factor (2:1 split on day 2); no factor on day 4, so it is not filled.
adj = pd.DataFrame({'close': [100.0, 50.0, 51.0, 52.0], 'adj_close': [50.0, np.nan, np.nan, np.nan]})
adj_fixed, used = rebuild_adj_close(adj, pd.Series([0.5, 1.0, 1.0, np.nan]), adj['adj_close'].isna())
assert adj_fixed['adj_close'].tolist()[:3] == [50.0, 50.0, 51.0] and np.isnan(adj_fixed.loc[3, 'adj_close'])
nb = neighbour_factor(pd.DataFrame({'ticker': list('AAABBB')}), pd.Series([0.5, 0.9, 0.5, 0.5, 0.9, 0.6]),
                      pd.Series([False, True, False, False, True, False]))
assert nb[1] == 0.5 and np.isnan(nb[4])

# Step 9: unknown volumes become flagged zeros.
vol = repair_volume(pd.DataFrame({'volume': [100.0, np.nan, -5.0, np.inf, 0.0]}))
assert vol['volume'].tolist() == [100.0, 0, 0, 0, 0] and vol['volume_repaired'].tolist() == [False, True, True, True, False]
print('Repair checks passed.')

,ticker,ts,open,high,low,close
0,A,2024-01-02,99.5,101.0,99.0,100.0
2,A,2024-01-04,109.5,111.0,109.0,110.0
3,B,2024-01-02,NaN,51.0,49.0,NaN
4,B,2024-01-03,50.0,52.0,49.5,51.0
5,C,2024-01-02,20.0,21.0,19.0,20.5
6,C,2024-01-03,22.0,22.5,21.5,22.5


Repair checks passed.


### 3.2 Investigating the extreme returns

Each flag of 2.4 gets one finding:

- `one_day_spike`: a move of 25% or more fully reversed the next session (within 5%). Treated as a data error and interpolated in step 6, even though the reference has the same values. The next session is `spike_reversal`.
- `corporate_action`: the reference factor steps up by more than 1% and stays there, and the adjusted return is not extreme. The raw drop comes from a dividend or split adjustment; the price is kept.
- `market_move`: the same move is in the reference raw and adjusted prices. Kept and flagged; the cause is not verified.

In [15]:
def investigate(rows):
    """One finding per 3-sigma flag."""
    g = rows.groupby('ticker')
    factor_ratio = rows['ref_factor'] / g['ref_factor'].shift(1)
    factor_stays = (g['ref_factor'].shift(-1) / rows['ref_factor'] - 1).abs().lt(0.005)
    adjusted_extreme = flag_extreme_returns(rows, price_col='ref_adj_close')['extreme']
    round_trip = (1 + rows['return']) * (1 + g['return'].shift(-1)) - 1
    spike = rows['extreme'] & rows['return'].abs().ge(0.25) & round_trip.abs().le(0.05)
    finding = np.select([spike, spike.groupby(rows['ticker']).shift(1, fill_value=False),
                         factor_ratio.gt(1.01) & factor_stays & ~adjusted_extreme],
                        ['one_day_spike', 'spike_reversal', 'corporate_action'], default='market_move')
    out = rows[['ticker', 'ts', 'close', 'volume', 'return']].assign(
        adjusted_return=rows['ref_adj_close'] / g['ref_adj_close'].shift(1) - 1, finding=finding)
    return out.loc[rows['extreme']]


findings = investigate(rows)
print(findings['finding'].value_counts().to_string())
display(findings.loc[findings['finding'] != 'market_move'])

finding
market_move         5045
corporate_action       5
one_day_spike          3
spike_reversal         3


,ticker,ts,close,volume,return,adjusted_return,finding
92142,CVX,1982-02-01,8.630000,1350000,-0.054765,-0.039735,corporate_action
192020,JPM,2002-10-02,18.250000,15141200,-0.075481,-0.059501,corporate_action
264448,MSFT,2004-11-15,27.390000,104734064,-0.086086,0.018717,corporate_action
318332,TRV,1976-10-28,9.220000,0,1.000000,2.273973,one_day_spike
318333,TRV,1976-10-29,4.690000,79200,-0.491323,-0.689386,spike_reversal
318873,TRV,1978-12-20,8.030000,0,0.997512,2.088973,one_day_spike
318874,TRV,1978-12-21,4.090000,105600,-0.490660,-0.669968,spike_reversal
319538,TRV,1981-08-10,12.060000,0,1.000000,1.675159,one_day_spike
319539,TRV,1981-08-11,6.030000,24800,-0.500000,-0.626190,spike_reversal
345722,VZ,1987-12-24,15.190000,1386821,-0.047051,-0.031915,corporate_action


The largest remaining moves, with the median close and volume over the 60 sessions after the move divided by the 60 sessions before:

In [16]:
def level_change(frame, i, col, n=60):
    window = frame.loc[max(i - n, 0): i + n - 1]
    window = window.loc[window['ticker'] == frame.at[i, 'ticker'], col]
    return window.loc[i:].median() / window.loc[:i - 1].median()


largest = findings.loc[findings['finding'] == 'market_move']
largest = largest.loc[largest['return'].abs().sort_values(ascending=False).index[:10]].copy()
largest['price after/before'] = [level_change(rows, i, 'close') for i in largest.index]
largest['volume after/before'] = [level_change(rows, i, 'volume') for i in largest.index]
display(largest.round(3))

,ticker,ts,close,volume,return,adjusted_return,finding,price after/before,volume after/before
213788,MCD,1968-01-02,0.175,6755143,-0.754,-0.756,market_move,0.266,4.056
4999,AAPL,2000-09-29,0.460,3128875900,-0.519,-0.519,market_move,0.353,1.897
280845,NVDA,2000-03-07,0.174,1952246199,0.424,0.424,market_move,2.047,1.479
281954,NVDA,2004-08-06,0.079,5092064120,-0.352,-0.352,market_move,0.681,1.436
336480,UNH,2008-10-13,23.380,17691904,0.348,0.347,market_move,0.810,0.973
22610,AMZN,2001-11-26,0.610,1013784000,0.345,0.345,market_move,1.552,1.557
331189,UNH,1987-10-20,0.098,2646400,-0.342,-0.342,market_move,0.609,1.516
22454,AMZN,2001-04-09,0.559,468114000,0.336,0.336,market_move,1.235,0.976
4204,AAPL,1997-08-06,0.235,4192008288,0.332,0.332,market_move,1.342,1.459
281641,NVDA,2003-05-09,0.178,9276606399,0.331,0.331,market_move,1.708,1.380


MCD on 1968-01-02 stands out: the price falls by about 4 and stays there while volume rises by about 4, and the adjustment factor does not move. That fits a split the vendor did not adjust for, but the data cannot confirm it or give the ratio, so the prices are not rescaled. The row is flagged `unexplained_level_shift` and returns across that date should not be used. The other moves are normal trading days and stay `market_move`.

In [17]:
findings.loc[findings['ticker'].eq('MCD') & findings['ts'].eq(pd.Timestamp('1968-01-02')), 'finding'] = 'unexplained_level_shift'

### 3.3 Cleaning

In [18]:
steps, quarantined = [], []


def record(step, rows_in, data, removed=0, set_aside=0):
    steps.append({'step': step, 'rows in': rows_in, 'removed': removed, 'quarantined': set_aside, 'rows out': len(data)})


# 1. keys
work = prepare_keys(raw)
rows_in = len(work)
bad_key = work['ts'].isna() | ~work['ticker'].isin(ref_tickers)
quarantined.append(work.loc[bad_key].assign(reason='unparseable date or unknown ticker'))
work = work.loc[~bad_key]
record('1. keys', rows_in, work, set_aside=int(bad_key.sum()))

# 2. exact duplicates
rows_in = len(work)
exact = work.duplicated(KEY + VALUES)
work = work.loc[~exact]
record('2. exact duplicates', rows_in, work, removed=int(exact.sum()))

# 3. conflicting keys
rows_in = len(work)
work, conflicts = resolve_conflicts(work, compare_with_reference(work, reference))
quarantined.append(conflicts.assign(reason='conflicting versions, none matches the reference'))
record('3. conflicting keys', rows_in, work, removed=rows_in - len(work) - len(conflicts), set_aside=len(conflicts))

# 4. rows without any raw price
rows_in = len(work)
empty = work[RAW_PRICES].isna().all(axis=1)
work = work.loc[~empty].sort_values(KEY)
record('4. empty rows', rows_in, work, removed=int(empty.sum()))

# 5. invalid and missing raw prices
rows_in = len(work)
work, invalid = invalidate_bad_prices(work)
work, filled = fill_missing_prices(work)
leading = work[RAW_PRICES].isna().any(axis=1)
display(work.loc[leading, KEY + RAW_PRICES])
quarantined.append(work.loc[leading].assign(reason='leading gap: no earlier price'))
work = work.loc[~leading]
repaired = (invalid | filled).loc[work.index]
record('5. invalid and missing raw prices', rows_in, work, set_aside=int(leading.sum()))

# 6. one-day spikes
rows_in = len(work)
spikes = findings.loc[findings['finding'] == 'one_day_spike']
spike_keys = set(zip(spikes['ticker'], spikes['ts']))
is_spike = pd.Series([key in spike_keys for key in zip(work['ticker'], work['ts'])], index=work.index)
before = work.loc[is_spike, KEY + RAW_PRICES]
work = interpolate_errors(work, pd.DataFrame({col: is_spike for col in RAW_PRICES}))
display(before.merge(work.loc[is_spike, KEY + RAW_PRICES], on=KEY, suffixes=('', '_repaired')))
edge = work[RAW_PRICES].isna().any(axis=1)
quarantined.append(work.loc[edge].assign(reason='error at the edge of the history'))
work = work.loc[~edge]
is_spike = is_spike.loc[work.index]
repaired = repaired.loc[work.index] | pd.DataFrame({col: is_spike for col in RAW_PRICES})
record('6. one-day spikes', rows_in, work, set_aside=int(edge.sum()))

,ticker,ts,open,high,low,close
287343,CVX,1962-01-02,NaN,3.30000,3.24000,3.30000
350808,JNJ,1962-01-02,NaN,0.22338,0.22222,0.22338
224790,MCD,1966-07-05,NaN,0.27366,0.26749,0.26955
364568,MRK,1962-01-02,NaN,0.39703,0.39317,0.39537
99637,PG,1962-01-02,NaN,1.44000,1.42000,1.43000


,ticker,ts,open,high,low,close,open_repaired,high_repaired,low_repaired,close_repaired
0,TRV,1976-10-28,9.22,9.22,9.22,9.22,4.650,4.650,4.650,4.650
1,TRV,1978-12-20,8.03,8.03,8.03,8.03,4.055,4.055,4.055,4.055
2,TRV,1981-08-10,12.06,12.06,12.06,12.06,6.030,6.030,6.030,6.030


In [19]:
# 7. remaining impossible OHLC rows
work, widened = widen_ranges(work)
repaired['high'] |= widened
repaired['low'] |= widened
record('7. impossible OHLC rows', len(work), work)
print(f'Rows widened: {int(widened.sum())}')

# 8. adjusted close
rows_in = len(work)
ref_view = compare_with_reference(work, reference)
factor = ref_view['ref_factor'].where(~is_spike, neighbour_factor(work, ref_view['ref_factor'], is_spike))
need = work['adj_close'].isna() | ref_view['factor_differs'] | repaired['close']
work, rebuilt = rebuild_adj_close(work, factor, need)
work['adj_close_repaired'] = rebuilt
no_factor = need & ~rebuilt
quarantined.append(work.loc[no_factor].assign(reason='no adjustment factor for this date'))
work = work.loc[~no_factor]
record('8. adjusted close', rows_in, work, set_aside=int(no_factor.sum()))
print(f'adj_close rebuilt: {int(rebuilt.sum()):,} rows')

# 9. volume
work = repair_volume(work)
record('9. volume', len(work), work)

work['raw_price_repaired'] = repaired.loc[work.index].any(axis=1)
finding_by_key = dict(zip(zip(findings['ticker'], findings['ts']), findings['finding']))
work['extreme_return_finding'] = [finding_by_key.get(key, '') for key in zip(work['ticker'], work['ts'])]
cleaned = work.sort_values(KEY).reset_index(drop=True)
quarantined = pd.concat(quarantined)
display(pd.DataFrame(steps))

Rows widened: 0
adj_close rebuilt: 28,793 rows


,step,rows in,removed,quarantined,rows out
0,1. keys,397711,0,0,397711
1,2. exact duplicates,397711,29460,0,368251
2,3. conflicting keys,368251,0,0,368251
3,4. empty rows,368251,0,0,368251
4,5. invalid and missing raw prices,368251,0,5,368246
5,6. one-day spikes,368246,0,0,368246
6,7. impossible OHLC rows,368246,0,0,368246
7,8. adjusted close,368246,0,0,368246
8,9. volume,368246,0,0,368246


## 4. Validate and save

In [20]:
def validate(frame):
    """Required properties of the cleaned table."""
    p, v = frame[PRICES], frame['volume']
    factor_ok = (~compare_with_reference(frame, reference)['factor_differs']
                 | frame['extreme_return_finding'].eq('one_day_spike'))  # spike days use the neighbours' factor
    return pd.Series({
        'no missing price': bool(p.notna().all().all()),
        'finite prices': bool(np.isfinite(p).all().all()),
        'positive prices': bool(p.gt(0).all().all()),
        'unique (ticker, ts)': not frame.duplicated(KEY).any(),
        'finite nonnegative volume': bool(np.isfinite(v).all() and v.ge(0).all()),
        'low <= open <= high': bool(frame['open'].between(frame['low'], frame['high']).all()),
        'low <= close <= high': bool(frame['close'].between(frame['low'], frame['high']).all()),
        'adj_close consistent with the reference factor': bool(factor_ok.all()),
    })


validation = validate(cleaned)
display(validation.to_frame('passed'))
assert validation.all()
after = price_checks(cleaned).sum()
print('Checks rerun on the cleaned table:', after[after > 0].to_dict())

,passed
no missing price,True
finite prices,True
positive prices,True
"unique (ticker, ts)",True
finite nonnegative volume,True
low <= open <= high,True
low <= close <= high,True
adj_close consistent with the reference factor,True


Checks rerun on the cleaned table: {'zero_volume': 74}


In [21]:
def profile(frame, tickers, dates):
    return pd.Series({
        'rows': len(frame),
        'ticker labels': frame['ticker'].nunique(),
        'repeated keys': int(pd.DataFrame({'ticker': tickers, 'ts': dates}).duplicated().sum()),
        'missing adj_close': int(frame['adj_close'].isna().sum()),
        'nonpositive prices': int(frame[PRICES].le(0).any(axis=1).sum()),
        'OHLC bound violations': int((~(frame['open'].between(frame['low'], frame['high'])
                                        & frame['close'].between(frame['low'], frame['high']))).sum()),
        'zero volume': int(frame['volume'].eq(0).sum()),
    })


display(pd.DataFrame({'before': profile(raw, raw_tickers, raw_dates),
                      'after': profile(cleaned, cleaned['ticker'], cleaned['ts'])}))
display(pd.concat({'before': raw[VALUES].describe().T, 'after': cleaned[VALUES].describe().T}, axis=1)
        .loc[:, (slice(None), ['count', 'mean', 'min', 'max'])].round(4))

removed = sum(step['removed'] for step in steps)
print(f'raw {len(raw):,} = retained {len(cleaned):,} + removed {removed:,} + quarantined {len(quarantined)}')
assert len(raw) == len(cleaned) + removed + len(quarantined)
display(quarantined['reason'].value_counts())

,before,after
rows,397711,368246
ticker labels,150,30
repeated keys,29460,0
missing adj_close,19855,0
nonpositive prices,5,0
OHLC bound violations,5,0
zero volume,81,74


before                                         after                                    
              count          mean     min           max     count          mean     min           max
open       397711.0  4.628550e+01  0.0000  6.042600e+02  368246.0  4.630940e+01  0.0052  6.042600e+02
high       397711.0  4.674770e+01  0.0052  6.086300e+02  368246.0  4.677180e+01  0.0052  6.086300e+02
low        397711.0  4.581630e+01  0.0048  5.978800e+02  368246.0  4.584030e+01  0.0048  5.978800e+02
close      397711.0  4.629290e+01  0.0052  6.054000e+02  368246.0  4.631730e+01  0.0052  6.054000e+02
adj_close  377856.0  5.165230e+01  0.0029  1.044251e+04  368246.0  3.727470e+01  0.0029  6.054000e+02
volume     397711.0  3.097773e+07  0.0000  9.276606e+09  368246.0  3.095184e+07  0.0000  9.276606e+09

raw 397,711 = retained 368,246 + removed 29,460 + quarantined 5


reason
leading gap: no earlier price    5
Name: count, dtype: int64

In [22]:
FLAGS = ['raw_price_repaired', 'adj_close_repaired', 'volume_repaired', 'zero_volume', 'conflict_resolved',
         'extreme_return_finding']
cleaned.assign(ts=cleaned['ts'].dt.strftime('%Y-%m-%d'))[['ts', 'ticker'] + VALUES + FLAGS] \
    .to_sql('ohlc_hw_cleaned', conn, if_exists='replace', index=False)

saved = pd.read_sql_query('SELECT * FROM ohlc_hw_cleaned', conn)
assert len(saved) == len(cleaned) and saved['ts'].str.fullmatch(r'\d{4}-\d{2}-\d{2}').all()
pd.testing.assert_frame_equal(pd.read_sql_query('SELECT * FROM ohlc_hw', conn), raw)
print(f'ohlc_hw_cleaned: {len(saved):,} rows saved; ohlc_hw unchanged.')
conn.close()

ohlc_hw_cleaned: 368,246 rows saved; ohlc_hw unchanged.
